# 02 — Prompt template development (PubMed)

## Rationale

- **`pubmed_t1` / `pubmed_t2`:** Original inline legend `A=BACKGROUND … E=CONCLUSIONS` + short instruction. Real runs (`mvp_pubmed_reliabili_*`) showed **systematic collapse to `A`/`BACKGROUND`** (see `docs/data_inventory.md` → BACKGROUND-collapse).
- **`pubmed_t3`:** Numbered category list **decouples letter order from semantic order**; maps categories 1→5 to A→E explicitly to probe whether the model follows numeric structure vs first-letter priming.
- **`pubmed_t4`:** Legend first, then a **direct question** (“Which single label letter…”) to vary instruction framing while keeping canonical `{legend}`.
- **`pubmed_t5`:** **Intentionally non-alphabetic legend order** with `A=BACKGROUND` **last** in the list to reduce **first-line / first-class-code priming** toward BACKGROUND.

## Mock render smoke (no GPU)
Below: each template rendered for one tiny fixture sentence; `mock_score_example` verifies the pipeline shape.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from reliability_eval.inference.score_class_codes import mock_score_example
from reliability_eval.prompting.label_codes import get_label_codes
from reliability_eval.prompting.render import render

cfg_dir = str(ROOT / "configs")
tiny_path = ROOT / "data" / "samples" / "pubmed_rct_tiny.jsonl"
row = json.loads(tiny_path.read_text(encoding="utf-8").splitlines()[0])
text = row["text"].strip()
codes = get_label_codes("pubmed_rct")

for tid in ("pubmed_t1", "pubmed_t2", "pubmed_t3", "pubmed_t4", "pubmed_t5"):
    prompt = render(
        template_id=tid,
        task="pubmed_rct",
        text=text,
        label_codes=codes,
        config_dir=cfg_dir,
    )
    m = mock_score_example(
        prompt=prompt,
        task="pubmed_rct",
        example_id=f"nb_{tid}",
        true_label=row.get("label"),
    )
    print("===", tid, "===")
    print(prompt[:400] + ("…" if len(prompt) > 400 else ""))
    print("mock ->", m["predicted_code"], m["confidence"])